# 多策略回测：EBSW 多阈值交叉

一个 `Bt` 里挂**多个策略子类**跑同一份数据，对比各自的绩效表格。本篇演示 `addstrategy()` 的用法与"多策略一次回测"的输出长什么样。

**本次演示**

- `Bt(auto=False)` → 只跑本文件 addstrategy 的策略
- 多个策略共享同一份 K 线，绩效表分开输出
- `bt.run()` 结束后的账户 / 统计表怎么读

**前置条件**：`pip install minibt`（本页只用**内置数据**，
不需要联网 / 账号，也不会下单）。终端若为 GBK，先设 `PYTHONIOENCODING=utf-8`。

**对应文档**：[1.7minibt_bt_class_intro.md](../minibt_docs/docs/minibt_basic/1.7minibt_bt_class_intro.md)


## 1. 导入与内置数据


In [ ]:
from minibt import *


class Strategy1(Strategy):
    params = dict(l1=.6, l2=.95, l3=.65, symbol="l2609_60")

    def __init__(self):
        self.min_start_length = 300
        self.data = self.get_kline(
            LocalDatas.get(self.params.symbol), height=300)
        self.gc = self.data.tradingview.G_Channels()
        self.ebsw = self.data.close.ebsw()
        self.macd = self.data.close.macd()
        self.adosc = self.data.close.ao()
        self.btindicatordataset.height=50
        self.buy_signal = self.ebsw.cross_up(self.params.l1)
        self.exit_buy_signal = self.ebsw.cross_up(
            self.params.l2) | self.ebsw.cross_down(self.params.l3)
        self.sell_signal=self.ebsw.cross_down(-self.params.l1)
        self.exit_sell_signal = self.ebsw.cross_down(
            -self.params.l2) | self.ebsw.cross_up(-self.params.l3)
        self.buy_signal.isplot=self.exit_buy_signal.isplot=self.sell_signal.isplot=self.exit_sell_signal.isplot=False

    def next(self):
        if not self.data.position:
            if self.buy_signal.new:
                self.data.buy()
            elif self.sell_signal.new:
                self.data.sell()
        elif self.data.position > 0 and self.exit_buy_signal.new:
            self.data.sell()
        elif self.data.position<0 and self.exit_sell_signal.new:
            self.data.buy()



# 单策略回测
if __name__ == "__main__":
    Bt(auto=False).addstrategy(Strategy1).run(isplot=True,isreport=True,  report_cwd="./test", report_name="test",gui=Gui.PylightchartsWeb)


## 2. 多策略回测 + 绩效表


In [ ]:
# 多策略回测
Strategy2 = Strategy1.copy(params=dict(l1=.5, l2=.8, l3=.7, symbol="pp2609_60"))
if __name__ == "__main__":
    Bt(auto=False).addstrategy(Strategy1,Strategy2).run(isplot=True,isreport=True,  report_cwd="./test", report_name="test",gui=Gui.PylightchartsWeb)

## 小结

- `auto=False` 时不会自动扫描导入的策略，便于对比实验；
- 每个策略的成交 / 权益是独立账户，互不影响。

## 下一步

- 想把结论画进桌面端：见 [1.33 Gui.Desk](../minibt_docs/docs/minibt_basic/1.33minibt_desk_gui.md)；
- 想找其它例子：看 [`tutorials/README.md`](../README.md) 的目录表；
- 有疑问 / 想补充：[提 Issue](https://github.com/MiniBtMaster/minibt/issues)。
